[Lab README](README.md)

# Lab 2.1: Vector retrieval, with and without the graph

This is the comparison the workshop is built on, run twice over the same index
with the same embedding.

`VectorRetriever` is standard RAG. It embeds the question, finds the nearest
chunks, and returns their text. Nothing else. Every vector database does this,
and for a paraphrased question about a fact stated in one chunk, it is enough.

`VectorCypherRetriever` starts identically, with the same vector search over the
same index, and then follows the matched chunk into the graph. The retrieved
text is the same. What comes back with it is not.

Run both. The difference is the point of the lab, and it is worth seeing rather
than being told.

In [ ]:
# At an AWS event: dependencies are pre-installed. Run this cell as-is.
# Self-paced: uncomment the line below first.
# !pip install -r requirements.txt

print("Environment ready")

## Connect and verify the graph

Retrieval notebooks do not create schema artifacts. This cell requires both Lab 1
indexes to be online with the expected label, property, dimensions, and
similarity function, then checks the graph facts the questions below depend on.

In [ ]:
import os

import boto3
from dotenv import load_dotenv
from IPython.display import HTML, display
from neo4j import GraphDatabase

load_dotenv()

NEO4J_VARS = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
missing = [name for name in NEO4J_VARS if not os.environ.get(name)]
has_aws = boto3.Session().get_credentials() is not None
RETRIEVAL_READY = not missing and has_aws

if missing:
    print(f"Neo4j is not configured: set {', '.join(missing)} in the repo-root .env")
if not has_aws:
    print("No AWS credentials found, so the query embedding cannot be computed")

if RETRIEVAL_READY:
    # Imported here because `workshop.graph_connection` raises at import when
    # NEO4J_PASSWORD is unset, which would fail this cell instead of skipping it.
    from workshop.graph_connection import NEO4J_URI, neo4j_auth
    from workshop.retrieval_contract import (
        CHUNK_FULLTEXT_INDEX,
        CHUNK_VECTOR_INDEX,
        EMBEDDING_DIMENSIONS,
        EMBEDDING_MODEL_ID,
    )
    from workshop.retrieval_setup import fixture_problems, verify_retrieval_indexes

    driver = GraphDatabase.driver(NEO4J_URI, auth=neo4j_auth())
    driver.verify_connectivity()

    try:
        verify_retrieval_indexes(driver)
        problems = fixture_problems(driver)
        if problems:
            raise RuntimeError("; ".join(problems))
    except Exception as exc:
        raise RuntimeError(
            f"The graph is not ready for retrieval: {exc}\n"
            "Run Lab 1 (01-graph-build/1.1_build_graph.ipynb) first."
        ) from exc

    print(f"{CHUNK_VECTOR_INDEX} is ONLINE")
    print(f"{CHUNK_FULLTEXT_INDEX} is ONLINE")
    print("Every graph fact these questions depend on is present")
else:
    print("\nThe cells below will skip. Finish Lab 0 and Lab 1, then come back.")

## The schema the graph actually holds

Lab 1 pinned this schema during extraction, so the traversals below can name
relationships instead of discovering them. Render it before using any retriever
that traverses, so the shape of the added context is predictable.

In [ ]:
from workshop.graph_schema import GRAPH_SCHEMA

pattern_rows = "".join(
    f"<tr><td><strong>{source}</strong></td><td>&mdash;{relationship}&rarr;</td>"
    f"<td><strong>{target}</strong></td></tr>"
    for source, relationship, target in GRAPH_SCHEMA["patterns"]
)
display(HTML(
    "<table><thead><tr><th>From</th><th>Relationship</th><th>To</th></tr>"
    f"</thead><tbody>{pattern_rows}</tbody></table>"
    "<p>Each extracted entity also points to its source "
    "<code>(entity)-[:FROM_CHUNK]-&gt;(:Chunk)</code>.</p>"
))

## One embedder, shared with the build

A query embedding has to come from the same model, at the same dimension, with
the same purpose as the vectors Lab 1 wrote. A mismatch does not raise. It
returns confident, wrong neighbours. So the embedder is constructed from the
same module the build used rather than configured again here.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from workshop.bedrock_providers import BedrockEmbeddings

    embedder = BedrockEmbeddings(region_name=os.environ.get("AWS_REGION", "us-east-1"))
    print(f"model: {EMBEDDING_MODEL_ID}")
    print(f"dimensions: {EMBEDDING_DIMENSIONS}")

    def show_results(question, result, why):
        """Print each retrieved item with its score, then why the pattern fits."""
        print(f"Question: {question}\n")
        for number, item in enumerate(result.items, 1):
            score = (item.metadata or {}).get("score")
            score_text = "n/a" if score is None else f"{score:.4f}"
            content = str(item.content)
            preview = content[:700] + ("…" if len(content) > 700 else "")
            print(f"[{number}] score={score_text}\n{preview}\n")
        print(f"Why this fits: {why}")

## Pattern 1: plain vector retrieval

Use this when the question is a paraphrase and the answer sits in the text of a
single chunk. The question below never says "policy", and the source document
never says "standard check-in time", so exact matching would find nothing.
Semantic similarity does.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from neo4j_graphrag.retrievers import VectorRetriever

    vector_retriever = VectorRetriever(
        driver=driver,
        index_name=CHUNK_VECTOR_INDEX,
        embedder=embedder,
        return_properties=["text"],
    )
    vector_question = (
        "What is the standard check-in time at the AnyCompany Cairo Nile View hotel?"
    )
    vector_result = vector_retriever.search(query_text=vector_question, top_k=3)
    show_results(
        vector_question,
        vector_result,
        "Semantic similarity finds the policy wording even when the question is paraphrased.",
    )

## Pattern 2: the same search, plus what the chunk is connected to

`VectorCypherRetriever` runs the identical vector search and then hands each
match to a Cypher query you wrote. The traversal below walks from the chunk to
the hotel it came from, then out to that hotel's rooms, amenities, policies, and
services.

The Cypher is static and reviewed. No part of it is generated by a model, and
nothing from the question is interpolated into it. That matters more than it
looks: this is the shape you can put behind a tool boundary and still reason
about what it can touch.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from neo4j_graphrag.retrievers import VectorCypherRetriever
    from neo4j_graphrag.types import RetrieverResultItem

    retrieval_query = """
    MATCH (hotel:Hotel)-[:FROM_CHUNK]->(node)
    OPTIONAL MATCH (hotel)-[relationship]->(detail)
    WHERE type(relationship) IN [
        'HAS_ROOM', 'OFFERS_AMENITY', 'HAS_POLICY', 'PROVIDES_SERVICE'
    ]
    WITH node, score, hotel,
         collect(DISTINCT {
             relationship: type(relationship),
             name: coalesce(detail.name, detail.type),
             description: detail.description
         })[..12] AS related
    RETURN node.text AS chunk, score,
           hotel { .name, .address, .guest_rating } AS hotel,
           related
    """

    def graph_result_formatter(record):
        """Keep the chunk, the hotel, and the traversal as separate fields."""
        content = {
            "chunk": record.get("chunk"),
            "hotel": record.get("hotel"),
            "related": record.get("related"),
        }
        return RetrieverResultItem(
            content=str(content),
            metadata={"score": record.get("score")},
        )

    vector_cypher_retriever = VectorCypherRetriever(
        driver=driver,
        index_name=CHUNK_VECTOR_INDEX,
        retrieval_query=retrieval_query,
        embedder=embedder,
        result_formatter=graph_result_formatter,
    )
    graph_question = (
        "Tell me about the hotel at 789 Avenue des Champs-Élysées and its amenities."
    )
    graph_result = vector_cypher_retriever.search(query_text=graph_question, top_k=2)
    show_results(
        graph_question,
        graph_result,
        "Vector search locates the chunk; Cypher adds the connected entities and typed relationships.",
    )

## What changed

Both retrievers found the same chunks, because they ran the same vector search.
Only the second one came back with the hotel's rating, its amenity list, and its
policies as named fields rather than as prose an answering model would have to
re-read and re-extract.

That is the whole delta, and it decides two things later in the workshop. An
agent that gets `guest_rating: 4.5` as a field cannot round it to 4 or invent a
5. And a write path that gets `hotel_id` back can act on the right hotel without
matching on a display name.

| Ask this | Reach for | Because |
|---|---|---|
| A paraphrased fact stated in one chunk | `VectorRetriever` | Meaning matters more than wording, and there is nothing to traverse |
| The same, plus the entity's connected facts | `VectorCypherRetriever` | One vector hop finds the chunk, one reviewed traversal finds the rest |

**Next:** `2.2_fulltext_retrievers.ipynb` takes on the question vector search is
worst at, an exact identifier, and builds the retriever Lab 5 deploys unchanged.

In [ ]:
if RETRIEVAL_READY:
    driver.close()
    print("Connection closed.")